# 03. 샘플링 프로토콜 ①②③ (10/27) — 최지훈 A 업무

| 프로토콜 | 내용 | 저장 파일 (`processed/{ds}/candidates/`) |
|---|---|---|
| ① | 사용자가 train·valid·test 어디에서도 평가하지 않은 상품 중 무작위 99개 + 정답 1개 | `p1_valid.parquet`, `p1_test.parquet` |
| ② | 정답과 같은 인기도 구간(train 평점 수의 log2 구간)에서 99개 + 정답 1개 | `p2_valid.parquet`, `p2_test.parquet` |
| ③ | 후보 없이 전체 상품 랭킹. 고정한 사용자 10,000명 (valid·test 같은 사용자) | `p3_users.parquet` |

- 후보 파일 컬럼: `uidx, iidx, label` (사용자당 100행, label=1이 정답). 정답 위치가 드러나지 않게 행 순서를 섞어 둠
- 함께 저장: `processed/{ds}/item_popularity.parquet` (item_id, iidx, train_count, bucket) → 공격 담당에게도 전달
- 시드: `SEED`와 (프로토콜, 분할) 번호를 묶어서 고정 → 누가 언제 돌려도 같은 후보
- 동점 처리: `src/eval/ranking.py`의 `target_rank`, `target_rank_full`

In [ ]:
# ===== 0. 설정 =====
import os, sys, subprocess, json, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/team10-recsys'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/qhfl3644/team10-recsys.git', REPO], check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
except ImportError:
    REPO = os.environ.get('TEAM10_REPO', '..')
sys.path.insert(0, REPO)
from config import *

import numpy as np
import pandas as pd
from src.data import sampling as S
from src.eval.ranking import target_rank, target_rank_full, hr_ndcg_at_k

N_P3_USERS = 10_000
SPLITS = ['valid', 'test']

In [ ]:
# ===== 1. 후보 파일 생성 =====
def build_candidates(ds):
    t0 = time.time()
    sp, n_users, n_items = S.load_indexed_splits(PROC_DIR, ds)
    indptr, indices = S.user_item_csr([sp['train'], sp['valid'], sp['test']], n_users)
    pop = S.item_popularity(sp['train'], n_items)
    bucket = S.popularity_bucket(pop)

    out_dir = f'{PROC_DIR}/{ds}/candidates'
    os.makedirs(out_dir, exist_ok=True)
    _, imap = S.load_maps(PROC_DIR, ds)
    pd.DataFrame({'item_id': imap['item_id'], 'iidx': imap['iidx'], 'train_count': pop, 'bucket': bucket}) \
      .to_parquet(f'{PROC_DIR}/{ds}/item_popularity.parquet', index=False)

    meta = {'dataset': ds, 'seed': SEED, 'n_users': n_users, 'n_items': n_items,
            'bucket_rule': 'train_count 0 -> 0, else floor(log2(count)) + 1',
            'bucket_sizes': {int(b): int(c) for b, c in zip(*np.unique(bucket, return_counts=True))}}
    for split_id, split in enumerate(SPLITS):
        target = sp[split]
        c1 = S.sample_protocol1(target, indptr, indices, n_items, np.random.default_rng([SEED, 1, split_id]))
        c1.to_parquet(f'{out_dir}/p1_{split}.parquet', index=False)
        c2, ranges = S.sample_protocol2(target, indptr, indices, bucket, np.random.default_rng([SEED, 2, split_id]))
        c2.to_parquet(f'{out_dir}/p2_{split}.parquet', index=False)
        meta[f'p2_{split}_expanded_users_pct'] = round(float((ranges[:, 0] != ranges[:, 1]).mean() * 100), 2)
        print(f'  {ds} {split}: ① {len(c1):,}행, ② {len(c2):,}행 ({time.time()-t0:.0f}s)')

    users3 = S.select_protocol3_users(n_users, N_P3_USERS, np.random.default_rng([SEED, 3]))
    umap, _ = S.load_maps(PROC_DIR, ds)
    pd.DataFrame({'uidx': users3, 'user_id': umap['user_id'].values[users3]}) \
      .to_parquet(f'{out_dir}/p3_users.parquet', index=False)
    meta['p3_n_users'] = int(len(users3))
    with open(f'{out_dir}/meta.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=1)
    print(f'=== {ds} 완료 ({time.time()-t0:.0f}s) ===')
    return dict(sp=sp, n_users=n_users, n_items=n_items, indptr=indptr, indices=indices,
                pop=pop, bucket=bucket, users3=users3, meta=meta)

ctx = {ds: build_candidates(ds) for ds in DATASETS}

In [ ]:
# ===== 2. 후보 파일 검증 =====
def check(label, ok):
    print(f'  [{"OK" if ok else "확인 필요"}] {label}')
    return ok

def verify_candidates(ds):
    c = ctx[ds]
    n_users, n_items = c['n_users'], c['n_items']
    rated_key = np.repeat(np.arange(n_users, dtype=np.int64), np.diff(c['indptr'])) * n_items + c['indices']
    ok = True
    print(f'=== {ds} ===')
    for proto in ['p1', 'p2']:
        for split in SPLITS:
            df = pd.read_parquet(f'{PROC_DIR}/{ds}/candidates/{proto}_{split}.parquet')
            g = df.groupby('uidx')
            target = c['sp'][split].set_index('uidx')['iidx']
            pos = df[df.label == 1].set_index('uidx')['iidx']
            neg = df[df.label == 0]
            neg_key = neg.uidx.values.astype(np.int64) * n_items + neg.iidx.values
            tag = f'{proto}_{split}'
            ok &= check(f'{tag}: 사용자 {df.uidx.nunique():,}명 × 100행', len(df) == n_users * 100 and (g.size() == 100).all())
            ok &= check(f'{tag}: 사용자마다 정답 1개, {split} 정답과 일치',
                        (g.label.sum() == 1).all() and pos.sort_index().equals(target.sort_index()))
            ok &= check(f'{tag}: 사용자 안에서 상품 중복 없음', not df.duplicated(['uidx', 'iidx']).any())
            ok &= check(f'{tag}: 오답 중 사용자가 평가한 상품 0개', not np.isin(neg_key, rated_key).any())
            if proto == 'p2':
                same = (c['bucket'][neg.iidx.values] == c['bucket'][np.repeat(pos.sort_index().values, 99)]).mean()
                print(f'      정답과 같은 인기도 구간인 오답 비율 {same*100:.1f}% '
                      f'(구간을 넓힌 사용자 {c["meta"][f"p2_{split}_expanded_users_pct"]}%)')
    # 재현성: 같은 시드로 앞 300명을 다시 뽑으면 후보 집합이 같아야 함
    head = c['sp']['valid'].head(300)
    again = S.sample_protocol1(head, c['indptr'], c['indices'], n_items, np.random.default_rng([SEED, 1, 0]))
    saved = pd.read_parquet(f'{PROC_DIR}/{ds}/candidates/p1_valid.parquet')
    saved = saved[saved.uidx.isin(head.uidx)]
    same = (again.sort_values(['uidx', 'iidx'])[['uidx', 'iidx']].values ==
            saved.sort_values(['uidx', 'iidx'])[['uidx', 'iidx']].values).all()
    ok &= check('재현성: 같은 시드로 다시 뽑으면 같은 후보', bool(same))
    ok &= check(f'③ 사용자 {len(c["users3"]):,}명, 중복 없음', len(np.unique(c['users3'])) == len(c['users3']))
    return bool(ok)

VERIFIED = {ds: verify_candidates(ds) for ds in DATASETS}
VERIFIED

## 3. 점검: 인기도 기준 모델과 "점수가 모두 같은" 모델

- **모든 점수가 같은 모델**: 동점 처리가 공정하면 ①②에서 HR@10 ≈ 10/100 = 0.10, ③에서 ≈ 10/상품 수
- **인기도 기준 모델** (점수 = log(1 + train 평점 수)): 제안서 예상대로라면 ①에서 가장 높고, ②에서 크게 떨어지고, ③에서 가장 낮아야 한다

여기서 계산한 HR·NDCG는 점검용이다. 최종 수치는 평가 담당의 지표 함수로 다시 계산한다.

In [ ]:
# ===== 3. 점검 =====
def ranks_sampled(df, score_of_item, rng):
    df = df.sort_values('uidx', kind='stable')
    s = score_of_item[df.iidx.values].reshape(-1, 100)
    t = (df.label.values == 1).reshape(-1, 100)
    return target_rank(s, t, rng)

def ranks_full(c, split, score_all, rng):
    target = c['sp'][split].set_index('uidx')['iidx']
    out = np.empty(len(c['users3']), dtype=np.int64)
    for r, u in enumerate(c['users3']):
        t = int(target[u])
        out[r] = target_rank_full(score_all, t, S.protocol3_exclude(c['indptr'], c['indices'], u, t), rng)
    return out

rows = []
for ds in DATASETS:
    c = ctx[ds]
    models = {'popularity': np.log1p(c['pop']).astype(np.float64), 'constant': np.zeros(c['n_items'])}
    for split in SPLITS:
        for proto in ['p1', 'p2', 'p3']:
            for name, score in models.items():
                rng = np.random.default_rng([SEED, 99])
                if proto == 'p3':
                    r = ranks_full(c, split, score, rng)
                else:
                    r = ranks_sampled(pd.read_parquet(f'{PROC_DIR}/{ds}/candidates/{proto}_{split}.parquet'), score, rng)
                hr, ndcg = hr_ndcg_at_k(r, 10)
                rows.append({'dataset': ds, 'split': split, 'protocol': proto, 'model': name,
                             'HR@10': round(hr, 4), 'NDCG@10': round(ndcg, 4)})
check_df = pd.DataFrame(rows)
os.makedirs(RESULT_DIR, exist_ok=True)
check_df.to_csv(f'{RESULT_DIR}/sampling_sanity_check.csv', index=False)
check_df.pivot_table(index=['dataset', 'model', 'split'], columns='protocol', values='HR@10')

## 확인할 것

1. 2번 셀이 모두 `[OK]`
2. 3번 표에서 `constant` 행: p1·p2 ≈ 0.10, p3 ≈ 0에 가까움 → 동점 처리가 정답에 유리하지 않다는 뜻
3. `popularity` 행: p1 > p2 > p3 순서 → 제안서 2.1절의 "①은 인기도 모델을 과대평가한다"가 실데이터에서 확인됨 (PPT 소재)
4. 평가 담당에게 넘길 것: `candidates/` 폴더, `src/eval/ranking.py`, `item_popularity.parquet`